In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
simulator = Aer.get_backend('aer_simulator')

In [2]:
# Alice's angles (0, pi/4) and Bob's angles (pi/8, 3pi/8)
angles = {
    'A0': 0, 
    'A1': np.pi/2, 
    'B0': np.pi/4, 
    'B1': 3*np.pi/4
}

def apply_measurement_rotation(qc, qubit, angle):
    """Applies a rotation to choose the measurement basis."""
    qc.ry(-angle, qubit) # Ry gate rotates the basis
    return qc

In [3]:
def apply_shared_entangle(qc, q0, q1):
    """The core entangling circuit, reused for all four Bell states."""
    qc.h(q0)
    qc.cx(q0, q1)
    return qc

In [4]:

def create_chsh_circuit(angle_a, angle_b):
    """Creates a quantum circuit for the CHSH game with given angles."""
    qc = QuantumCircuit(2, 2)  # 2 qubits and 2 classical bits for measurement results
    apply_shared_entangle(qc, 0, 1)  # Create the shared entangled state

    # Alice's measurement
    qc.barrier()
    apply_measurement_rotation(qc, 0, angle_a)  # Rotate Alice's qubit based on her angle
    qc.measure(0, 0)  # Measure Alice's qubit

    # Bob's measurement
    qc.barrier()
    apply_measurement_rotation(qc, 1, angle_b)  # Rotate Bob's qubit based on his angle
    qc.measure(1, 1)  # Measure Bob's qubit

    return qc

In [5]:
def get_expectation(counts):
    """Calculates <A*B> from measurement counts."""
    shots = sum(counts.values())
    # Parity: +1 if results match (00, 11), -1 if they differ (01, 10)
    correlation = (counts.get('00', 0) + counts.get('11', 0) - 
                   counts.get('01', 0) - counts.get('10', 0)) / shots
    return correlation

# Calculate S = E(A0,B0) - E(A0,B1) + E(A1,B0) + E(A1,B1)
results = {}
for a_label in ['A0', 'A1']:
    for b_label in ['B0', 'B1']:
        qc = create_chsh_circuit(angles[a_label], angles[b_label])
        job = simulator.run(transpile(qc, simulator), shots=8192)
        results[(a_label, b_label)] = get_expectation(job.result().get_counts())

S = results[('A0', 'B0')] - results[('A0', 'B1')] + results[('A1', 'B0')] + results[('A1', 'B1')]
print(f"Calculated CHSH Score (S): {S:.4f}")

Calculated CHSH Score (S): 2.8250
